# 19 — Declared multi-seed check of the C_110 neural result

Notebook 16's headline — neural pooled 150.6 / neural FedAvg 152.4 vs persistence 154.1 on 1,401
identical C_110 rows — is one seed among 184 computed scores, so it was flagged single-seed and
unconfirmed. This notebook is the declared robustness check, approved in the 2026-07-30 ratification
(notebook 15, `docs/supervisor_decision_sheet.md`).

**Declared before running, nothing chosen after seeing results:** seeds **42, 123, 999**; both
regimes; both neural models; C_110 only. Implementation (window build, architecture, training
budget) is copied **verbatim from notebook 16** so the only varying quantity is the seed. Every seed
is reported, none is discarded, and no seed is selected for anything downstream. **Verdict rule,
declared a priori:** a model's sub-persistence claim is *confirmed* only if it beats persistence
under **all three** seeds in the full regime; one failure = *not robust*.

Two caveats stated up front: cuDNN training is not bit-reproducible, so seed 42 here lands near, not
exactly on, 16's stored values (the official numbers remain 16's csv); and the linear models are
exact least squares with no seed dependence, so they are read from `fl_neural_summary.csv`, not
retrained. This check runs in its own notebook rather than inside 16 so 16's stored csvs and the
report figures pinned to them are not perturbed by a nondeterministic re-run.

In [1]:
# Goal: the C_110 window set, built exactly as notebook 16 built it (code copied verbatim).
import numpy as np                           # window arrays and arithmetic guards
import pandas as pd                          # panel loading and the result table
import torch                                 # the GRU clients, retrained per declared seed
from torch import nn                         # layers and the MSE training loss
import copy                                  # deepcopy hands each client its own copy of the global model
from pathlib import Path

PROC = Path("../data/processed")

TEST_START = pd.Timestamp("2025-04-01")      # sealed test year - never read below
VAL_START = pd.Timestamp("2024-04-01")       # validation year FY 2024-25 (unchanged)
COVID = pd.Timestamp("2020-03-01")           # declared regime break (unchanged)

MS_SEEDS = [42, 123, 999]                    # declared 2026-07-30 before any run - all reported
SEQ_LEN = 6                                  # every constant below is 16's, unchanged
HIDDEN = 32
ROUNDS, LOCAL_EPOCHS = 10, 5
POOLED_EPOCHS = ROUNDS * LOCAL_EPOCHS
LR = 1e-3

device = torch.device("cuda")                # CUDA verified on this machine; a missing GPU crashes loudly
panel = pd.read_csv(PROC / "rtt_kh03_canonical_panel.csv", parse_dates=["month"])
dev = panel[(panel["month"] < TEST_START) & panel["fl_client"] & panel["is_public"]]

def build_windows(df):                       # verbatim from 16 - one specialty in, flat window set out
    df = df.sort_values(["provider_code", "month"]).copy()
    g = df.groupby("provider_code")["total_waiting"]
    df["target"] = g.shift(-1)               # next month's list
    df["delta_1"] = df["total_waiting"] - g.shift(1)
    df = df.dropna(subset=["target", "delta_1"])       # first row per provider has no momentum yet
    for c in ["pct_gt18", "pct_gt52"]:       # a zero list is 0/0 = NaN in the panel; an empty list
        df[c] = np.where(df["total_waiting"] > 0, df[c], 0.0)   # has no long-waiters, so the share is 0

    X, y, month, prov, current, target, d1 = [], [], [], [], [], [], []
    for p, r in df.groupby("provider_code"):
        f = np.column_stack([np.log1p(r["total_waiting"]), r["delta_1"] / 100,
                             r["pct_gt18"], r["pct_gt52"]]).astype(np.float32)
        t = ((r["target"] - r["total_waiting"]) / 100).to_numpy(np.float32)
        for i in range(SEQ_LEN - 1, len(r)):           # a provider shorter than SEQ_LEN adds nothing
            X.append(f[i - SEQ_LEN + 1:i + 1]); y.append(t[i])
        month += list(r["month"].iloc[SEQ_LEN - 1:]); prov += [p] * (len(r) - SEQ_LEN + 1)
        current += list(r["total_waiting"].iloc[SEQ_LEN - 1:])
        target += list(r["target"].iloc[SEQ_LEN - 1:]); d1 += list(r["delta_1"].iloc[SEQ_LEN - 1:])
    return (np.stack(X), np.array(y, dtype=np.float32), pd.Series(month), np.array(prov),
            np.array(current, dtype=float), np.array(target, dtype=float), np.array(d1, dtype=float))

X, y, month, prov, current, target, d1 = build_windows(dev[dev["tfunc_code"] == "C_110"])
assert len(month) == 0 or month.max() < TEST_START     # the sealed-year sentinel, same as 16
val = (month >= VAL_START).to_numpy()
assert int(val.sum()) == 1401                # the identical eligible rows 16 scored - drift trips this
truth, cur = target[val], current[val]
mae_pers = float(np.abs(truth - cur).mean())
assert round(mae_pers, 1) == 154.1           # 16's stored persistence anchor on these rows
Xv = torch.tensor(X[val]).to(device)
print(f"C_110: {int(val.sum())} eligible validation windows | persistence MAE {mae_pers:.1f} (n=1,401)")

C_110: 1401 eligible validation windows | persistence MAE 154.1 (n=1,401)


In [2]:
# Goal: the client model and both training procedures - verbatim from 16.
class DeltaGRU(nn.Module):                   # one GRU layer reads the 6 months; a linear head -> delta/100
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(4, HIDDEN, batch_first=True)
        self.head = nn.Linear(HIDDEN, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1]).squeeze(-1)      # the last month's hidden state carries the sequence

def train_steps(model, X, y, epochs):        # full-batch Adam: client datasets are tiny (<= ~90 windows)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    mse = nn.MSELoss()
    for _ in range(epochs):
        opt.zero_grad(); loss = mse(model(X), y); loss.backward(); opt.step()

def fit_fedavg(client_data):                 # classic FedAvg: broadcast, local steps, weight-average
    glob = DeltaGRU().to(device)
    for _ in range(ROUNDS):
        states, ns = [], []
        for Xc, yc in client_data:
            local = copy.deepcopy(glob)
            train_steps(local, Xc, yc, LOCAL_EPOCHS)
            states.append(local.state_dict()); ns.append(len(yc))
        n_tot = sum(ns)
        glob.load_state_dict({k: sum(sd[k] * n for sd, n in zip(states, ns)) / n_tot
                              for k in states[0]})    # sample-weighted average of every weight tensor
    return glob

def fit_pooled(X, y):                        # centralised reference on the identical total step budget
    model = DeltaGRU().to(device)
    train_steps(model, X, y, POOLED_EPOCHS)
    return model

In [3]:
# Goal: 12 scores (3 seeds x 2 regimes x 2 models) -> fl_neural_c110_multiseed.csv + the declared verdict.
rows = []
for seed in MS_SEEDS:
    for regime, in_reg in [("full", ~val), ("post", ~val & (month >= COVID).to_numpy())]:
        pr = prov[in_reg]
        Xt = torch.tensor(X[in_reg]).to(device); yt = torch.tensor(y[in_reg]).to(device)
        cd = [(Xt[pr == p], yt[pr == p]) for p in np.unique(pr)]
        torch.manual_seed(seed)              # identical init per model, exactly as 16 seeds its models
        npool = fit_pooled(Xt, yt)
        torch.manual_seed(seed)
        nfed = fit_fedavg(cd)
        with torch.no_grad():                # x100 undoes the declared target scaling
            preds = {"neural_pooled": cur + 100 * npool(Xv).cpu().numpy(),
                     "neural_fedavg": cur + 100 * nfed(Xv).cpu().numpy()}
        for mname, pred in preds.items():
            err = truth - pred
            rows.append({"seed": seed, "regime": regime, "model": mname,
                         "mae": float(np.abs(err).mean()),
                         "rmse": float(np.sqrt((err ** 2).mean())), "n": int(val.sum())})
        print(f"seed {seed} {regime}: pooled {rows[-2]['mae']:.1f}, fedavg {rows[-1]['mae']:.1f} "
              f"(persistence {mae_pers:.1f}, n=1,401)")

ms = pd.DataFrame(rows)
ms.to_csv(PROC / "fl_neural_c110_multiseed.csv", index=False)

print()
for regime in ["full", "post"]:
    for mname in ["neural_pooled", "neural_fedavg"]:
        s = ms[(ms["regime"] == regime) & (ms["model"] == mname)]["mae"]
        beats = int((s < mae_pers).sum())
        print(f"{regime:5s} {mname:14s} MAE {s.mean():.1f} +/- {s.std():.1f} "
              f"(seeds: {', '.join(f'{v:.1f}' for v in s)}) | beats persistence {beats} of 3 seeds")

# the verdict rule declared in the intro: confirmed only if all three full-regime seeds beat persistence
for mname in ["neural_pooled", "neural_fedavg"]:
    s = ms[(ms["regime"] == "full") & (ms["model"] == mname)]["mae"]
    verdict = "CONFIRMED (all 3 seeds below persistence)" if (s < mae_pers).all() else "NOT ROBUST"
    print(f"declared verdict, full regime, {mname}: {verdict}")
print(ms.shape, "->", PROC / "fl_neural_c110_multiseed.csv")

C:\Users\heave\Documents\NHS UK FL+Agentic AI\nhs_FL_agentic\Lib\site-packages\torch\nn\modules\rnn.py:1449: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  result = _VF.gru(


seed 42 full: pooled 150.6, fedavg 152.4 (persistence 154.1, n=1,401)


seed 42 post: pooled 151.4, fedavg 153.7 (persistence 154.1, n=1,401)


seed 123 full: pooled 152.2, fedavg 152.3 (persistence 154.1, n=1,401)


seed 123 post: pooled 154.1, fedavg 153.4 (persistence 154.1, n=1,401)


seed 999 full: pooled 151.9, fedavg 152.4 (persistence 154.1, n=1,401)


seed 999 post: pooled 153.0, fedavg 153.6 (persistence 154.1, n=1,401)

full  neural_pooled  MAE 151.5 +/- 0.9 (seeds: 150.6, 152.2, 151.9) | beats persistence 3 of 3 seeds
full  neural_fedavg  MAE 152.4 +/- 0.0 (seeds: 152.4, 152.3, 152.4) | beats persistence 3 of 3 seeds
post  neural_pooled  MAE 152.8 +/- 1.4 (seeds: 151.4, 154.1, 153.0) | beats persistence 2 of 3 seeds
post  neural_fedavg  MAE 153.6 +/- 0.2 (seeds: 153.7, 153.4, 153.6) | beats persistence 3 of 3 seeds
declared verdict, full regime, neural_pooled: CONFIRMED (all 3 seeds below persistence)
declared verdict, full regime, neural_fedavg: CONFIRMED (all 3 seeds below persistence)
(12, 6) -> ..\data\processed\fl_neural_c110_multiseed.csv
